In [16]:
import requests
import pandas as pd
from time import sleep

LIST_URL = ("https://api.geckoterminal.com/api/v2/tokens/info_recently_updated?network=solana")
DETAIL_URL = ("https://api.geckoterminal.com/api/v2/networks/solana/tokens/{address}/info")


def fetch_recent_addresses(limit):
    resp = requests.get(LIST_URL)
    resp.raise_for_status()
    data = resp.json().get("data", [])
    addresses = [item["attributes"]["address"] for item in data]
    return addresses[:limit]


def fetch_token_info(address):
    resp = requests.get(DETAIL_URL.format(address=address))
    resp.raise_for_status()
    attr = resp.json().get("data", {}).get("attributes", {})
    top10 = None
    gt_score = attr.get("gt_score")
    holders = attr.get("holders", {}).get("distribution_percentage", {})
    if "top_10" in holders:
        top10 = float(holders["top_10"])

    return {
        "address": address,
        "t10_percent": top10,
        "pass_report": gt_score,
    }


def fetch_dex_info(address):
    url = f"https://api.dexscreener.com/token-pairs/v1/solana/{address}"
    resp = requests.get(url)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return {
            "txns_h24": None,
            "volume_h24": None,
            "volume_m5": None,
            "price_h24": None,
            "price_m5": None,
            "liquidity_usd": None,
            "fdv": None,
        }
    first = data[0]
    if(first.get("liquidity",0)!=0):
        liq = first.get("liquidity").get("usd")
    else:
        liq = 0
    return {
        "txns_h24": (first["txns"]["h24"]["buys"] + first["txns"]["h24"]["sells"]),
        "volume_h24": first["volume"].get("h24"),
        "volume_m5": first["volume"].get("m5"),
        "price_h24": first["priceChange"].get("h24"),
        "price_m5": first["priceChange"].get("m5",0),
        "liquidity_usd": liq,
        "fdv": first.get("fdv"),
    }


def main():
    addresses = fetch_recent_addresses(200)
    rows = []

    for i, addr in enumerate(addresses, 1):
        try:
            token_data = fetch_token_info(addr)
            dex_data = fetch_dex_info(addr)
            combined = {**token_data, **dex_data}
            rows.append(combined)
        except Exception as e:
            print(f"[{i}/{len(addresses)}] error for {addr}: {e}")
        sleep(0.2)
        if i % 29 == 0:
            print(f"[{i}/{len(addresses)}] batch complete, pausing 62 seconds...")
            sleep(62)
    columns = [
        "address",
        "t10_percent",
        "txns_h24",
        "volume_h24",
        "volume_m5",
        "price_h24",
        "price_m5",
        "liquidity_usd",
        "fdv",
        "pass_report",
    ]
    df = pd.DataFrame(rows, columns=columns)
    df.to_excel("input_tokens.xlsx", index=False)
    print(f"Wrote {len(df)} records to input_tokens.xlsx")


if __name__ == "__main__":
    main()


[29/100] batch complete, pausing 62 seconds...
[58/100] batch complete, pausing 62 seconds...
[87/100] batch complete, pausing 62 seconds...
Wrote 100 records to input_tokens.xlsx
